In [ ]:
import pandas as pd
import numpy as np
import re

From https://www.wormatlas.org/neurotransmitterstable.htm

**Acetylcholine**

ADF(2), AIA(2), AIM(2)4, AIN(2), AIY(2), ALN(2), AS1-11, ASJ(2), AVA(2), AVB(2), AVD(2), AVE(2), AVG5, AWB(2), CA1-9(m)4, CEM(4, m), DA1-9, DB1-7,  DVA6, DVE(m), DX1(m)7, DX2(m)7, HOB(m), HSN(2, h), I1(2)5, I35, IL2(6), M15, M2(2)5, M4, M5, MC(2)5, PCB(2, m), PCC(2, m), PDA, PDB, PDC(m)8, PGA(m)8, PLN(2), PVC(2), PVN(2)5, PVP(2), PVV(m), PVX(m), PVY(m), PVZ(m), R1A(2, m), R2A(2, m), R3A(2, m), R4A(2, m), R6A(2, m), RIB(2)8, RIF(2), RIH, RIR, RIV(2), RMD(6), RMF(2), RMH(2), SAA(4)5, SAB(3), SDQ(2), SIA(4), SIB(4), SMB(4), SMD(4), SPC(2, m), SPV(2, m), URA(4), URB(2), URX(2), VA1-12, VB1-11, VC1-6(h) 

**GABA**

AVL, CP9(m), DD1-6, DVB, EF1-4(m), R2A(m), R6A(m), R9B(m), RIB(2), RIS, RME, SMDD/V, VD1-13

**Glutamate**

ADA(2), ADL(2), AFD(2), AIB(2), AIM(2)5, AIZ(2),  ALM(2), ASE(2), ASG(2)38, ASH(2), ASK(2), AQR, AUA(2), AVM, AWC(2), BAG(2), CP0, CP5-7, DVC, FLP(2),  HOA(m), I2(2), I543, IL1(6), LUA(2), M3(2), MI, OLL(2), OLQ(4),  PCA(2, m) PHA(2), PHB(2)43, PHC(2), PLM(2), PVD(2), PVQ(2), PQR(2), PVR, PVV(m), R2B(2, m), R5A(2, m), R6B(2, m), R9A(2, m), RIA(2), RIG(2), RIM(2), URY(4)

Executive decisions made:
- **RIB is inhibitory (GABA)**, see footnote on NT table and Tsalik and Hobert, 2003
- **SMDs are excitatory (acetylcholine)**, see [Yeon et al. 2018](https://doi.org/10.1371/journal.pbio.2004929)

AIM is truly a dual cholin- and glutamatergic neuron

It seems that glutamate is presumed **excitatory** in *C. elegans*.

GABA seems to inhibitory except that it is contracting, not relaxing, in the motor neurons AVL and DVB (which control defecation I think).

I4, I6, ASI, PVM, BDU, AVF, AVH, RMG, AVK, AVJ, PVT, PVW, RID, and CAN have [unknown or no classical NT](https://www.wormatlas.org/neurotransmitterstable.htm#Neurons%20lacking%20assignnents). Some have known functions, others don't. NSM is seroton- and peptidergic only. ADE, CEP is dopamin- and peptidergic only. I am zeroing out these rows in this version of the data.

[Wang et al. 2024](https://doi.org/10.7554/eLife.95402.2) reports AWA, PDE, RIC to be glutamatergic and RIP to be cholinergic so I went with that to zero out fewer rows. They also predict that ALA is able to synaptically release GABA that it takes up from other cells.

In [ ]:
# load raw data
w = pd.read_excel("SI5-302.xlsx",sheet_name=1,index_col=0)
w.fillna(0,inplace=True)
nrns = w.columns

# connectivity data
achNrns = ["ADF", "AIA", "AIM", "AIN", "AIY", "ALN", "AS\d", "ASJ", "AVA", "AVB", "AVD", "AVE", "AVG", "AWB", "CA\d", "CEM", "DA\d", "DB\d", "DVA", "DVE", "DX1", "DX2", "HOB", "HSN", "I1", "I3", "IL2", "M1", "M2", "M4", "M5", "MC", "PCB", "PCC", "PDA", "PDB", "PDC", "PGA", "PLN", "PVC", "PVN", "PVP", "PVV", "PVX", "PVY", "PVZ", "R1A", "R2A", "R3A", "R4A", "R6A", "RIF", "RIH", "RIR", "RIV", "RMD", "RMF", "RMH", "SAA", "SAB", "SDQ", "SIA", "SIB", "SMB", "SMD", "SPC", "SPV", "URA", "URB", "URX", "VA\d", "VB\d", "VC\d", "RIP"]
negGabaNrns = ["DD\d", "EF\d", "R2A", "R6A", "R9B", "RIB", "RIS", "RME", "VD\d", "ALA"]
posGabaNrns = ["AVL", "DVB"]
glutNrns = ["ADA", "ADL", "AFD", "AIB", "AIM", "AIZ", "ALM", "ASE", "ASG", "ASH", "ASK", "AQR", "AUA", "AVM", "AWC", "BAG", "DVC", "FLP", "HOA", "I2", "I5", "IL1", "LUA", "M3", "MI", "OLL", "OLQ", "PCA", "PHA", "PHB", "PHC", "PLM", "PVD", "PVQ", "PQR", "PVR", "PVV", "R2B", "R5A", "R6B", "R9A", "RIA", "RIG", "RIM", "URY", "AWA", "PDE", "RIC"]
nullNrns = ["I4", "I6", "ASI", "PVM", "BDU", "AVF", "AVH", "RMG", "AVK", "AVJ", "PVT", "PVW", "RID", "CAN", "NSM", "ADE", "CEP"]

# make GABA neurons inhibitory and null neurons 0
w.loc[nrns[[any([re.match(regex,nrn) for regex in negGabaNrns]) for nrn in nrns]]] = w.loc[nrns[[any([re.match(regex,nrn) for regex in negGabaNrns]) for nrn in nrns]]] * -1
w.loc[nrns[[any([re.match(regex,nrn) for regex in nullNrns]) for nrn in nrns]]] = w.loc[nrns[[any([re.match(regex,nrn) for regex in nullNrns]) for nrn in nrns]]] * 0

# save
# w.to_csv("connectivity.csv")

In [ ]:
# Save accompanying types into csv
nrnTypes = pd.read_excel("SI5-302.xlsx",sheet_name=0,index_col=0)
nrnTypes = pd.Series(nrnTypes.iloc[:,0])
nrnTypes.to_csv("neuron_types.csv",index_label="name")

In [ ]:
# plot, if you want

import seaborn as sns
from matplotlib import colors

sns.heatmap(w,norm = colors.SymLogNorm(5,vmin=-75,vmax=75),
                    cmap=sns.blend_palette(["#026b85","#00acd6","white","#ff9985","#d84a2e"],as_cmap=True),square=True,
                    linewidths=0.25, linecolor=None,
                    annot=False) #,fmt=".0f",clip_on=False,cbar=False,cbar_kws={"fraction":0.05,"label":"synaptic weight"},annot_kws={"fontsize":6,"color":connColorMap(0.5)})